<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Import-Data" data-toc-modified-id="Import-Data-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Import Data</a></span></li><li><span><a href="#Phenotype-Table" data-toc-modified-id="Phenotype-Table-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Phenotype Table</a></span><ul class="toc-item"><li><span><a href="#Update-Phenotype" data-toc-modified-id="Update-Phenotype-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Update Phenotype</a></span></li><li><span><a href="#Intersect-with-BCR-dataset" data-toc-modified-id="Intersect-with-BCR-dataset-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Intersect with BCR dataset</a></span></li></ul></li></ul></div>

## Import Data

In [12]:
import pandas as pd
import sys,os
sys.path.append('/proj/regeps/regep00/studies/COPDGene/analyses/remge/PUMA_final/src/')
from config import Config
from utils_puma import get_processed_copdgene

config = Config('bulk')

path_output = config.path_output
dict_input = get_processed_copdgene(config.path_dict_processed_input)

#df_exp, ,df_pheno_mirna,df_exp_mirna,list_case,list_control = dict_input['df_exp'], dict_input['df_pheno'], dict_input['df_pheno_mirna'],dict_input['df_exp_mirna'],dict_input['list_case'],dict_input['list_control']

#set_mirna,df_motif,dict_mirna_merging = dict_input['set_mirna'],dict_input['df_motif'],dict_input['dict_mirna_merging']


## Phenotype Table

In [2]:
from tableone import TableOne, load_dataset
df_pheno = dict_input['df_pheno'].copy()
selected_bool_var = ['gender','race','smoking_status_P2'] #,'finalGold_P2'
selected_cont_var = ['Age_P2','BMI_P2','FEV1_FVC_post_P2','FEV1_FVC_pre_P2', 'Duration_Smoking_P2']#'post_fev1pp',
df_pheno.finalGold_P2 = 'GOLD '+df_pheno.finalGold_P2.astype(int).astype(str)
df_pheno.race = df_pheno.race.replace({1:'White',2:'Black'})
df_pheno.gender = df_pheno.gender.replace({1:'Male',2:'Female'})
df_pheno.smoking_status_P2 = df_pheno.smoking_status_P2.replace({1:'Former',2:'Current'})
mytable = TableOne(df_pheno, columns=selected_cont_var+selected_bool_var, categorical=selected_bool_var, groupby=['finalGold_P2'],pval=True,
                  decimals={'FEV1_FVC_post_P2':2}) 
#nonnormal=nonnormal, rename=labels,
# chi squared for categorical, anova (t test) for cont, ranking for nonnormal
mytable.tableone.columns = [x[1] for x in mytable.tableone.columns]
mytable.tableone = mytable.tableone.drop(['Missing','Overall'],axis=1)
#mytable.tableone = mytable.tableone.rename(columns={'0':'Controls','1':'Cases'})

# remove _P2
mytable.tableone.index = mytable.tableone.index.set_levels(
    mytable.tableone.index.levels[0].str.replace('_P2', '').str.replace('_', ' ', regex=False),
    level=0
)

mytable

GOLD 0       GOLD 1       GOLD 2       GOLD 3       GOLD 4 P-Value
n                                           1607          366          708          361          148        
Age, mean (SD)                        63.4 (8.3)   68.8 (8.6)   67.6 (8.6)   69.1 (8.1)   67.6 (7.7)  <0.001
BMI, mean (SD)                        29.2 (6.0)   26.6 (4.6)   28.8 (6.3)   27.8 (6.6)   26.0 (6.1)  <0.001
FEV1 FVC post, mean (SD)             0.78 (0.05)  0.65 (0.04)  0.59 (0.08)  0.44 (0.09)  0.34 (0.08)  <0.001
FEV1 FVC pre, mean (SD)                0.8 (0.1)    0.6 (0.1)    0.6 (0.1)    0.4 (0.1)    0.3 (0.1)  <0.001
Duration Smoking, mean (SD)          34.9 (11.2)  39.7 (11.7)  40.9 (10.5)   42.7 (9.8)   41.0 (8.9)  <0.001
gender, n (%)               Female    851 (53.0)   154 (42.1)   320 (45.2)   151 (41.8)    62 (41.9)  <0.001
                            Male      756 (47.0)   212 (57.9)   388 (54.8)   210 (58.2)    86 (58.1)        
race, n (%)                 Black     507 (31.5)    74 (20.2)   158 (22.3)    71 (19.7)    26 (17.6)  <0.001
                            White    1100 (68.5)   292 (79.8)   550 (77.7)   290 (80.3)   122 (82.4)        
smoking status, n (%)       Current   614 (38.2)   139 (38.0)   281 (39.7)    97 (26.9)    24 (16.2)  <0.001
                            Former    993 (61.8)   227 (62.0)   427 (60.3)   264 (73.1)   124 (83.8)

In [9]:
mytable.tableone.loc['FEV1 FVC pre, mean (SD)','GOLD 0']

    0.8 (0.1)
Name: GOLD 0, dtype: object

In [40]:
import pyreadr

file_miRNA_pheno =  '/proj/regeps/regep00/studies/COPDGene/analyses/rebdh/craigMiRna/mirSeqPheno_filt_blockFreeze2_20200608.rds'
df_pheno_0 = pyreadr.read_r(config.path_mirna_pheno)[None]

df_pheno_0 = df_pheno_0.set_index('sid')
df_pheno_0 = df_pheno_0.query('mirSeqBatch!="plate5"')

df_exp_pheno = pd.read_csv(config.path_mrna_pheno, sep ='\t',na_values=' ',index_col=0,low_memory=False)

In [32]:
(df_pheno_0['FEV1_FVC_utah_P2']-df_pheno['FEV1_FVC_post_P2']).dropna()

10052Z    0.0
10055F    0.0
10060Y    0.0
10076N    0.0
10086Q    0.0
         ... 
25674M    0.0
25818K    0.0
25822B    0.0
26021Y    0.0
26092V    0.0
Length: 382, dtype: float64

In [35]:
df_pheno.loc[df_pheno_0.index[df_pheno_0['FEV1_FVC_utah_P2'].isna()]]

KeyError: "None of [Index(['11704V', '14725S', '24123W', '10595L', '11807F', '21764T', '16489Q',\n       '19410S', '16134F', '13125P', '15520F'],\n      dtype='object', name='sid')] are in the [index]"

In [45]:
df_exp_pheno.loc[df_pheno_0.index[df_pheno_0['FEV1_FVC_utah_P2'].isna()],'FEV1_FVC_post_P2']

sid
11704V   NaN
14725S   NaN
24123W   NaN
10595L   NaN
11807F   NaN
21764T   NaN
16489Q   NaN
19410S   NaN
16134F   NaN
13125P   NaN
15520F   NaN
Name: FEV1_FVC_post_P2, dtype: float64

### Update Phenotype 

Using new definition of COPD:

"individuals with severe (FEV1<50% predicted and FEV1/FVC<0.7) or moderate (50%<FEV1<80% and FEV1/FVC<0.7) COPD, as well as control subjects (FEV1≥80% and FEV1/FVC≥0.7)."

COPD: FEV1<80% and FEV1/FVC<0.7 ; Control: FEV1>=80% and FEV1/FVC>=0.7; other: everyone else



In [15]:
list_case_new =  df_pheno.query('FEV1_FVC_post_P2<0.7 and FEV1pp_post_P2<80').index
list_control_new =  df_pheno.query('FEV1_FVC_post_P2>=0.7 and FEV1pp_post_P2>=80').index
df_pheno['COPD_new'] = 'Other'
df_pheno.loc[list_case_new,'COPD_new']='COPD'
df_pheno.loc[list_control_new,'COPD_new']='Control'

In [16]:
mytable = TableOne(df_pheno, columns=selected_cont_var+selected_bool_var, categorical=selected_bool_var, groupby='COPD_new',pval=True,
                  decimals={'FEV1_FVC_post_P2':2}) 


In [17]:
mytable

Grouped by COPD_new                                                            
                                                   Missing      Overall         COPD      Control        Other P-Value
n                                                                  3190         1217         1607          366        
Age_P2, mean (SD)                                        0   65.8 (8.7)   68.0 (8.4)   63.4 (8.3)   68.8 (8.6)  <0.001
BMI_P2, mean (SD)                                        0   28.5 (6.1)   28.2 (6.4)   29.2 (6.0)   26.6 (4.6)  <0.001
FEV1_FVC_post_P2, mean (SD)                              0  0.66 (0.15)  0.51 (0.12)  0.78 (0.05)  0.65 (0.04)  <0.001
FEV1_FVC_pre_P2, mean (SD)                               4    0.6 (0.1)    0.5 (0.1)    0.8 (0.1)    0.6 (0.1)  <0.001
Duration_Smoking_P2, mean (SD)                           3  37.9 (11.3)  41.4 (10.1)  34.9 (11.2)  39.7 (11.7)  <0.001
gender, n (%)                  Female                    0  1538 (48.2)   533 (43.8)   851 (53.0)   154 (42.1)  <0.001
                               Male                         1652 (51.8)   684 (56.2)   756 (47.0)   212 (57.9)        
race, n (%)                    Black                     0   836 (26.2)   255 (21.0)   507 (31.5)    74 (20.2)  <0.001
                               White                        2354 (73.8)   962 (79.0)  1100 (68.5)   292 (79.8)        
smoking_status_P2, n (%)       Current                   0  1155 (36.2)   402 (33.0)   614 (38.2)   139 (38.0)   0.014
                               Former                       2035 (63.8)   815 (67.0)   993 (61.8)   227 (62.0)

### Intersect with BCR dataset

In [7]:
import pyreadr
file_r = "/proj/regeps/regep00/studies/COPDGene/analyses/remol/immune_seq/pheno_airr2.Rdata"


df_bcr = pyreadr.read_r(file_r) # also works for Rds
df_bcr = df_bcr['pheno'].set_index('sid')


In [8]:
df_pheno_bcr = df_pheno.merge(df_bcr, left_index=True, right_index=True)

In [9]:
from tableone import TableOne, load_dataset
selected_bool_var = ['gender','race','finalGold_P2','smoking_status_P2']
selected_cont_var = ['Age_P2','BMI_P2','FEV1_FVC_post_P2','FEV1_FVC_pre_P2', 'Duration_Smoking_P2']#'post_fev1pp',
#df_pheno = df_pheno.query('finalGold_P2==0 or finalGold_P2>=2')
df_pheno['finalGold_P2_disc'] = (df_pheno['finalGold_P2']>=2).astype(int)
mytable = TableOne(df_pheno_bcr, columns=selected_cont_var+selected_bool_var, categorical=selected_bool_var, groupby=['finalGold_P2_disc','SmokCigNow_P2'],pval=True,
                  decimals={'FEV1_FVC_post_P2':2}) 
#nonnormal=nonnormal, rename=labels,
# chi squared for categorical, anova (t test) for cont, ranking for nonnormal
mytable.tableone.columns = [x[1] for x in mytable.tableone.columns]
mytable.tableone = mytable.tableone.drop(['Missing','Overall'],axis=1)
mytable.tableone = mytable.tableone.rename(columns={'0':'Controls','1':'Cases'})
mytable

Controls        Cases P-Value
n                                            78           59        
Age_P2, mean (SD)                    62.6 (6.9)   67.1 (8.1)   0.001
BMI_P2, mean (SD)                    27.9 (4.9)   27.9 (6.3)   0.961
FEV1_FVC_post_P2, mean (SD)         0.75 (0.08)  0.53 (0.12)  <0.001
FEV1_FVC_pre_P2, mean (SD)            0.7 (0.1)    0.5 (0.1)  <0.001
Duration_Smoking_P2, mean (SD)       41.0 (9.2)   45.9 (8.6)   0.002
gender, n (%)                  1      29 (37.2)    30 (50.8)   0.154
                               2      49 (62.8)    29 (49.2)        
race, n (%)                    1      55 (70.5)    47 (79.7)   0.309
                               2      23 (29.5)    12 (20.3)        
finalGold_P2, n (%)            0.0    60 (76.9)               <0.001
                               1.0    18 (23.1)                     
                               2.0                 32 (54.2)        
                               3.0                 23 (39.0)        
                               4.0                   4 (6.8)        
smoking_status_P2, n (%)       1.0    40 (51.3)    26 (44.1)   0.507
                               2.0    38 (48.7)    33 (55.9)        
[1] Chi-squared tests for the following variables may be invalid due to the low number of observations: finalGold_P2.